In [ ]:
import numpy as np
import cv2
import glob
import pickle
import matplotlib.pyplot as plt
import tools_camera as ca  # Assumo che tu abbia questo modulo

""""Extraction of pairs of world points and their corresponding projected points in pixels"""
# prepare object points, like (0,0,0), (1,0,0), ..., (8,5,0)
objp = np.zeros((6*9, 3), np.float32)
objp[:, :2] = np.mgrid[0:9, 0:6].T.reshape(-1, 2)  # (9,6) -> 54 points

# Arrays to store object points and image points from all the images.
objpoints = []  # 3d points in real world space
imgpoints = []  # 2d points in image plane.

# Make a list of calibration images
images = glob.glob('chessboard.bmp')
# oppure più immagini: glob.glob('imgs/calibration/left*.jpg')

# Step through the list and search for chessboard corners
for idx, fname in enumerate(images):
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Find the chessboard corners
    ret, corners = cv2.findChessboardCorners(gray, (9,6), None)  # attenzione a (9,6)

    # If found, add object points, image points
    if ret == True:
        objpoints.append(objp)
        imgpoints.append(corners)

        # Draw and display the corners
        cv2.drawChessboardCorners(img, (9,6), corners, ret)
        cv2.imshow('img', img)
        cv2.waitKey(500)

cv2.destroyAllWindows()

"""Camera calibration"""
# Do camera calibration given object points and image points
img_size = (img.shape[1], img.shape[0])
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, img_size, None, None)

# Compute Projection matrix for the first image
K = mtx
R = cv2.Rodrigues(rvecs[0])[0]
T = tvecs[0]
RT = np.concatenate((R, T), axis=1)
Pest0 = np.dot(K, RT)

print('Projection matrix for image0:')
print(ca.ndtotext(Pest0))

"""Save the camera calibration result for later use (we won't worry about rvecs / tvecs)"""
dist_pickle = {}
dist_pickle["mtx"] = mtx
dist_pickle["dist"] = dist
pickle.dump(dist_pickle, open("imgs/calibration_wide/wide_dist_pickle.p", "wb"))

"""Show error in the first image with OpenCV functionality"""
U = cv2.projectPoints(objpoints[0], rvecs[0], tvecs[0], mtx, dist)
error = np.asarray(U[0][:, 0] - imgpoints[0][:, 0, :])
error = np.linalg.norm(error) / len(error)
print("Retroprojection error with OpenCV for first image:", error)

# Show error in the first image with Python functionality
mpoints = np.transpose(objpoints[0])
n_points = len(mpoints[0])
lista_ones = np.ones((1, n_points)).tolist()
mpoints = np.append(mpoints, lista_ones, axis=0)

U = np.dot(Pest0, mpoints)
U = [[U[0] / U[2]], [U[1] / U[2]]]
U = np.reshape(np.array(U, dtype="float32").transpose(), (n_points, 1, 2))

imgpoints_nd = cv2.undistortPoints(imgpoints[0], mtx, dist, None, mtx)
error1 = U - imgpoints_nd
error1 = np.linalg.norm(error1) / len(error1)
print("Retroprojection error for first image:", error1)

""""Display an undistorted Test image"""
img = cv2.imread('imgs/calibration_wide/test_image.jpg')
dst = cv2.undistort(img, mtx, dist, None, mtx)

f, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax1.set_title('Original Image', fontsize=30)
ax2.imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB))
ax2.set_title('Undistorted Image', fontsize=30)
plt.show()
